# Retry/Wait Sandbox

### Logic

- Repeat until no exception
- Repeat until no exception, or N times, whichever comes first
- Repeat at most N times
- Repeat at most N times until the result satisfies a predicate
- N: -1 means forever

### Callable: Action

- func
- args
- kwargs
- Timed out value

### Other Callables: Predicate and Correction

- Context: object which stores
    - Last result, or
    - Last exception
    - N
    - Number of iterations so far
    - Action
        - func
        - args
        - kwargs
        - Timed out value
    - Other data, depends on the caller
- Predicate: takes a context, return bool
- Correction: takes a context, perform correction

### Delay

- Linear backup
- Exponential backup
- Custom backup


### Logging

- Default logger if not supplied
- Log level


In [14]:
import itertools
import dataclasses
from typing import Any, Union, Callable, Optional


@dataclasses.dataclass
class Context:
    res: Any
    exc: Exception
    func: callable
    args: list
    kwargs: dict
    n: int
    i: int
    func_timeout: float
    ignore: Union[None, Exception, tuple[Exception]]
    logger: logging.Logger


Corrector = Callable[[Context], None]
Predicate = Callable[[Context], bool]


def _default_corrector(ctx: Context):
    pass


def _default_predicate(ctx: Context):
    return True

    
def retry(
    func: Callable,
    args: Optional[tuple] = None,
    kwargs: Optional[dict] = None,
    func_timeout: Optional[float] = None,
    n: int,
    ignore: Union[None, Exception, tuple[Exception]] = None,
    predicate: Optional[Predicate] = None,
    corrector: Optional[Corrector] = None,
    logger: Optional[logging.Logger] = None,
):
    if predicate is None:
        predicate = _default_predicate
    if corrector is None:
        corrector = _default_corrector
    if logger is None:
        logger = logging.getLogger("retry")
        
    ctx = Context(res=None, exc=None, n=n, i=0, func=func, args=args, kwargs=kwargs, func_timeout=func_timeout, ignore=ignore)

In [15]:
import logging
logger = logging.getLogger()
logger

<RootLogger root (WARNING)>

In [16]:
type(logger)

logging.RootLogger

In [5]:
def linear_delay(duration):
    return itertools.repeat(duration)

In [6]:
d = linear_delay(5)

In [8]:
next(d)

5

In [17]:
lg = logging.getLogger("foo.bar")
lg

<Logger foo.bar (WARNING)>

In [18]:
type(lg)

logging.Logger